In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"
N_DIAS_ALVO = 30

fato_titulos = spark.table(f"{GOLD}.fato_titulos")

base = (
    fato_titulos
    .withColumn("dias_desde_vencimento",
                F.datediff(F.lit(DATA_CORTE).cast("date"), F.col("data_vencimento")))
    .withColumn("entra_treino",
                F.col("dias_desde_vencimento") > N_DIAS_ALVO)
    .withColumn("alvo",
                F.when(~F.col("entra_treino"), F.lit(None))
                 .when(F.col("dias_atraso") > N_DIAS_ALVO, F.lit(1))
                 .when(F.col("dias_atraso") <= N_DIAS_ALVO, F.lit(0)))
)

base.groupBy("entra_treino").count().show()
base.filter(F.col("entra_treino")).groupBy("status", "alvo").count().orderBy("status", "alvo").show()